# Algorithm Comparison

Compare DQN, PPO, and A2C performance on Ms. Pac-Man.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from src.utils import plot_comparison

## 1. Load Training Logs

This section assumes you have TensorBoard log files from training all three algorithms.

In [ ]:
from tensorboard.backend.event_processing import event_accumulator
import os

def load_tensorboard_data(log_dir, tag='episode/mean_reward_100'):
    """Load TensorBoard scalar data."""
    ea = event_accumulator.EventAccumulator(log_dir)
    ea.Reload()
    
    if tag in ea.Tags()['scalars']:
        events = ea.Scalars(tag)
        steps = [e.step for e in events]
        values = [e.value for e in events]
        return steps, values
    return [], []

In [ ]:
# Load data for each algorithm
results = {}

for algo in ['dqn', 'ppo', 'a2c']:
    log_dirs = [d for d in os.listdir('../logs') if d.startswith(algo)]
    if log_dirs:
        steps, values = load_tensorboard_data(f'../logs/{log_dirs[0]}')
        if values:
            results[algo.upper()] = values
            print(f'{algo.upper()}: {len(values)} data points')

print(f'\nLoaded data for {len(results)} algorithms')

## 2. Plot Learning Curves

In [ ]:
if results:
    plot_comparison(results, save_path='../logs/comparison.png', show=True)
else:
    print('No training data found. Train algorithms first using notebooks 02-04.')

## 3. Performance Comparison Table

In [ ]:
if results:
    print('Algorithm Performance Summary:\n')
    print(f"{'Algorithm':<10} {'Final (100ep)':<15} {'Max':<10} {'Convergence':<12}")
    print('-' * 50)
    
    for algo, rewards in results.items():
        if len(rewards) > 0:
            final = np.mean(rewards[-10:]) if len(rewards) >= 10 else rewards[-1]
            maximum = np.max(rewards)
            # Find step where reached 80% of max
            threshold = 0.8 * maximum
            conv_idx = next((i for i, r in enumerate(rewards) if r >= threshold), len(rewards))
            print(f"{algo:<10} {final:<15.1f} {maximum:<10.1f} {conv_idx * 1000:<12}")

else:
    print('No data to analyze')

## 4. Sample Efficiency Comparison

In [ ]:
if results:
    plt.figure(figsize=(12, 5))
    
    for algo, rewards in results.items():
        # Compute sample efficiency (reward per 1000 steps)
        steps = np.arange(len(rewards)) * 1000
        plt.plot(steps, rewards, label=algo, linewidth=2)
    
    plt.xlabel('Training Steps')
    plt.ylabel('Average Reward (100 episodes)')
    plt.title('Sample Efficiency Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 5. Stability Comparison

In [ ]:
if results:
    print('Training Stability (Standard Deviation):\n')
    for algo, rewards in results.items():
        if len(rewards) > 10:
            std = np.std(rewards)
            print(f'{algo}: {std:.2f}')

## Conclusions

- **DQN**: Sample efficient but slower to train
- **PPO**: Most stable and reliable
- **A2C**: Fastest updates but higher variance

Choose based on your priorities:
- Sample efficiency → DQN
- Stability → PPO
- Speed → A2C